# Multi-Label Regression with TFIDF
https://www.kaggle.com/code/lonnieqin/multi-label-regression-with-tfidf  
tf-idfで推論


## import

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error

## configuration

In [ ]:
class Config:
    vocab_size = 8192
    embed_size = int(vocab_size ** 0.5)  #embed_size?
    batch_size = 32
    epochs = 30
    use_k_fold = True
    target_columns = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
    dataset_path = "../input/feedback-prize-english-language-learning"
config = Config() 

## loading data

In [ ]:
train = pd.read_csv(f"{config.dataset_path}/train.csv")
test = pd.read_csv(f"{config.dataset_path}/test.csv")


In [ ]:
train.head()

## preprocessing

In [ ]:
# nltk.word_tokenize()で分かち書きしている
train["text"] = train["full_text"].apply(lambda sentence: " ". join(nltk.word_tokenize(sentence.lower())))
test["text"] = test["full_text"].apply(lambda sentence: " ". join(nltk.word_tokenize(sentence.lower())))

In [ ]:
# print(train['full_text'][0])
# print('------------------')
# print(train['text'][0])

In [ ]:
train_1 = train[0:3800]
train_2 = train[3800:]

In [ ]:
vectorizor = keras.layers.TextVectorization(
    max_tokens=config.vocab_size, 
    output_mode="tf-idf", 
    ngrams=1
)
vectorizor.adapt(list(train_1["text"]))

## building model

In [ ]:
def get_model():
    model = keras.Sequential([
        keras.Input(shape=(), dtype="string"),
        vectorizor,
        keras.layers.Dense(64, kernel_initializer='he_uniform', activation='swish'),
        keras.layers.Dense(32, kernel_initializer='he_uniform', activation='swish'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(len(config.target_columns))
    ])
    rmse = tf.keras.metrics.RootMeanSquaredError(name="rmse")
    model.compile(loss="mse", optimizer="adam", metrics=[rmse])
    return model

In [ ]:
model = get_model()
model.summary()

In [ ]:
keras.utils.plot_model(model, show_shapes=True)

In [ ]:
keras.backend.clear_session()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
models = []
rmses = []
for i, (train_indices, valid_indices) in enumerate(kfold.split(train_1)):
    x_train = train_1.iloc[train_indices]["text"]
    y_train = train_1.iloc[train_indices][config.target_columns]
    x_val = train_1.iloc[valid_indices]["text"]
    y_val = train_1.iloc[valid_indices][config.target_columns]
    model_path = f"model_{i}.tf"
    model = get_model()
    rmse = tf.keras.metrics.RootMeanSquaredError(name="rmse")
    checkpoint = keras.callbacks.ModelCheckpoint(model_path, monitor="val_rmse", mode="min", save_best_only=True, save_weights_only=True)
    early_stop = keras.callbacks.EarlyStopping(monitor="val_rmse", mode="min", patience=5)
    model.compile(loss="mse", optimizer="adam", metrics=[rmse])
    history = model.fit(
        x_train, y_train, 
        batch_size=config.batch_size, 
        epochs=config.epochs,
        validation_data=(x_val, y_val),
        callbacks=[checkpoint, early_stop]
    )
    model.load_weights(model_path)
    result = model.evaluate(x_val, y_val)
    print("Loss:", result[0], "RMSE:", result[1])
    rmses.append(result[1])
    models.append(model) 
    if not config.use_k_fold:
        break
print(f"Mean RMSE:{np.mean(rmses)}")

In [ ]:
preds = []
for model in models:
    preds.append(model.predict(test["text"]))
pred = np.mean(preds, axis=0)
submission = pd.DataFrame({
    "text_id": test["text_id"]
})
for i in range(len(config.target_columns)):
    column = config.target_columns[i]
    submission[column] = pred[:, i]
pred = np.mean(preds, axis=0)
submission.to_csv("submission.csv", index=False)


In [ ]:
submission

In [ ]:
preds = []
for model in models:
    preds.append(model.predict(train_2["text"]))
train2_pred = np.mean(preds, axis=0)

def RMSE(label, pred):
    return np.sqrt(mean_squared_error(label, pred))

rmse_value = []
for j in range(len(config.target_columns)): #6
    p_lists = []
    l_lists = []
    for i in range(len(train2_pred)): #111
        p = train2_pred[i].tolist()[j]
        l = train_2.iloc[i][config.target_columns].tolist()[j]
        p_lists.append(p)
        l_lists.append(l)
    rmse = RMSE(l_lists, p_lists)
    rmse_value.append(rmse)

In [ ]:
df_ = pd.DataFrame(data={
    'index': config.target_columns,
    'rmse': rmse_value
})
display(df_)

In [ ]:
vectorizor.get_vocabulary()